In [1]:

%pip install pyomo highspy

  Using cached pyomo-6.10.0-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [highspy]
Note: you may need to restart the kernel to use updated packages.


RESTART KERNEL!

Zweite Energiemodelle und Analysen Hausübung

NAMEN MTRKLNR.

Achmed Achmedowski
Raphael Elser
Rainer Enders 
Sara 

Aufgabenstellung 10:

Gegenstand dieser Arbeit ist die mathematische Modellierung und Optimierung der Produktionsstrategie eines Wasserstoffelektrolyseurs. Im Fokus steht dabei eine binäre Entscheidungsfindung: Basierend auf den fluktuierenden Produktionskosten (insbesondere Strompreisen), den vertraglich fixierten Erlösen sowie potenziellen Strafzahlungen bei Nichtlieferung muss entschieden werden, ob der tägliche Wasserstoffbedarf vollständig gedeckt oder die Produktion gänzlich eingestellt wird. Da eine Teilversorgung explizit ausgeschlossen ist, wird ein wirtschaftlicher Abwägungsprozess modelliert, der den ökonomisch optimalen Betriebszustand unter Berücksichtigung dieser Parameter ermittelt.

In [2]:
import pyomo.environ as pyo
import pandas as pd
import numpy as np

# 1. Daten generieren (Beispielwerte)
hours = range(24)
prices = [20, 15, 10, 12, 18, 30, 50, 80, 100, 90, 70, 60, 
          55, 65, 85, 120, 150, 140, 90, 60, 40, 30, 25, 20] # Strompreise in €/MWh

model = pyo.ConcreteModel()

# Sets & Parameter
model.T = pyo.Set(initialize=hours)
model.p_el = pyo.Param(model.T, initialize=dict(enumerate(prices)))
model.D_total = pyo.Param(initialize=500) # kg Bedarf
model.r_h2 = pyo.Param(initialize=15.0)   # €/kg Erlös
model.c_pen = pyo.Param(initialize=2000.0) # € Strafe
model.p_max = pyo.Param(initialize=2.0)    # MW Leistung
model.eta = pyo.Param(initialize=0.05)     # MWh/kg (ca. 50 kWh/kg)

# Variablen
model.x = pyo.Var(domain=pyo.Binary)
model.h = pyo.Var(model.T, domain=pyo.NonNegativeReals)

# Zielfunktion
def obj_rule(m):
    revenue = m.x * (m.r_h2 * m.D_total)
    costs = sum(m.p_el[t] * m.eta * m.h[t] for t in m.T)
    penalty = (1 - m.x) * m.c_pen
    return revenue - costs - penalty
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

# Nebenbedingungen
def demand_rule(m):
    return sum(m.h[t] for t in m.T) == m.x * m.D_total
model.con_demand = pyo.Constraint(rule=demand_rule)

def cap_rule(m, t):
    return m.h[t] <= m.x * (m.p_max / m.eta)
model.con_cap = pyo.Constraint(model.T, rule=cap_rule)

# Solver aufrufen (z.B. glpk oder cbc)
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(model)

# Ergebnisse ausgeben
print(f"Produktionsentscheidung (x): {pyo.value(model.x)}")
print(f"Gesamtgewinn: {pyo.value(model.obj)} €")


Produktionsentscheidung (x): 1.0
Gesamtgewinn: 6790.0 €
